# Bonus lecture: Data Shapes

### `{}` versus `[{}]`, and everything in between

Every dataset you will ever touch, from a CSV to an API response to a database row,
arrives as one of about eight shapes built out of two things: **dictionaries** and
**lists**.

Once you can look at a structure and say "that is a list of dictionaries", you
already know how to get any value out of it. This lecture is about building that
reflex.


---

## The only rule you need

Read the structure **from the outside in**, one bracket at a time.

| You see | It means | You use |
|---|---|---|
| `{` | a dictionary | a **key**: `thing["name"]` |
| `[` | a list | a **position** or a **loop**: `thing[0]`, or `for item in thing:` |

That is the whole trick. `[{...}]` starts with `[`, so it is a list. Loop it.
Each item inside is `{...}`, so it is a dictionary. Use keys on it.

```python
rows = [{"id": 1, "name": "Ada"}]
#      ^ list                        -> loop or index
#       ^ dict                       -> use a key

rows[0]["name"]
#   ^ position into the list
#      ^ key into the dictionary
```

In [175]:
rows = [{"id": 1, "name": "Ada"}, {"id": 2, "name": "Ogee"}]

---

## 1. Dict: one record

The simplest shape. One thing, with named fields.

In [176]:
d = {"id": 1, "name": "Ada", "age": 30} 

print(d)
print(type(d))
print(len(d), "fields")

{'id': 1, 'name': 'Ada', 'age': 30}
<class 'dict'>
3 fields


In [177]:
# Access one field
print(d["name"])
print(d["age"])

Ada
30


In [178]:
# Safe access, for a field that might not be there
print(d.get("email"))                    # None, no crash
print(d.get("email", "not provided"))    # your own fallback

# Compare with the unsafe version
try:
    print(d["email"])
except KeyError as e:
    print(f"KeyError: {e}")

None
not provided
KeyError: 'email'


In [179]:
# Select only some fields
selected = {}
for key in ["id", "name"]:
    selected[key] = d[key]

print(selected)

{'id': 1, 'name': 'Ada'}


In [180]:
# Modify a field, and add a new one
d["age"] = 31
d["country"] = "NG"

print(d)

{'id': 1, 'name': 'Ada', 'age': 31, 'country': 'NG'}


In [181]:
# Loop over a single record
for key, value in d.items():
    print(f"{key}: {value}")

id: 1
name: Ada
age: 31
country: NG


In [182]:
# As a DataFrame. Note the square brackets: pandas wants a LIST of records.
import pandas as pd

pd.DataFrame([d])

,id,name,age,country
0,1,Ada,31,NG


`pd.DataFrame(d)` without the brackets raises an error, because pandas expects
rows. One record is a list of one row, so wrap it: `pd.DataFrame([d])`.

In [183]:
try:
    pd.DataFrame(d)
except ValueError as e:
    print(f"ValueError: {e}")

print()
print("with the brackets:")
print(pd.DataFrame([d]))

ValueError: If using all scalar values, you must pass an index

with the brackets:
   id name  age country
0   1  Ada   31      NG


---

## 2. List of dict: rows

**This is the most important shape in data engineering.** It is a table.

- The **list** is the table.
- Each **dictionary** inside it is one row.
- The **keys** are the column names.

In [184]:
rows = [
    {"id": 1, "name": "Ada",   "age": 30, "city": "Lagos"},
    {"id": 2, "name": "Tunde", "age": 25, "city": "Abuja"},
    {"id": 3, "name": "Chi",   "age": 35, "city": "Lagos"},
]

print(f"{len(rows)} rows")
print(f"columns: {list(rows[0].keys())}")

3 rows
columns: ['id', 'name', 'age', 'city']


In [229]:
city = []
name = []
for i in range(3):
    city.append(rows[i]["city"])
    name.append(rows[i]["name"])
    print(rows[i]["name"], rows[i]["city"])

Ada Lagos
Tunde Abuja
Chi Lagos


In [230]:
city

['Lagos', 'Abuja', 'Lagos']

In [231]:
name

['Ada', 'Tunde', 'Chi']

In [232]:
city_list = []
name_list = []
for i in rows:
    city_list.append(i["city"])
    name_list.append(i["name"])
    #print([i]["name"], [i]["city"])

In [233]:
city_list

['Lagos', 'Abuja', 'Lagos']

In [234]:
name_list

['Ada', 'Tunde', 'Chi']

In [235]:
# Access one value: position into the list, then key into the dict
print(rows[0])              # the whole first row
print(rows[0]["name"])      # one field of the first row
print(rows[-1]["city"])     # last row

{'id': 1, 'name': 'Ada', 'age': 30, 'city': 'Lagos', 'age_next_year': 31}
Ada
Lagos


In [236]:
# Get one whole column
names = []
for row in rows:
    names.append(row["name"])

print(names)

['Ada', 'Tunde', 'Chi']


In [237]:
# Select some columns, every row
selected = []
for row in rows:
    small = {}
    for key in ["id", "name"]:
        small[key] = row[key]
    selected.append(small)

print(selected)

[{'id': 1, 'name': 'Ada'}, {'id': 2, 'name': 'Tunde'}, {'id': 3, 'name': 'Chi'}]


In [194]:
# Filter rows
older = []
for row in rows:
    if row["age"] > 26:
        older.append(row)

print(f"{len(older)} row(s) over 26")
for row in older:
    print("  ", row["name"], row["age"])

2 row(s) over 26
   Ada 30
   Chi 35


In [195]:
# Modify every row
for row in rows:
    row["age_next_year"] = row["age"] + 1

for row in rows:
    print(row["name"], row["age"], "->", row["age_next_year"])

Ada 30 -> 31
Tunde 25 -> 26
Chi 35 -> 36


Notice you did not need to reassign anything. `row` inside the loop **is** the
dictionary in the list, not a copy, so changing it changes the table.

That is convenient here and a trap elsewhere. If you want to leave the original
alone, build a new list instead.

In [196]:
# Leaving the original untouched
original = [{"id": 1, "age": 30}, {"id": 2, "age": 25}]

changed = []
for row in original:
    new_row = dict(row)              # a copy of this row
    new_row["age"] = row["age"] + 1
    changed.append(new_row)

print("changed: ", changed)
print("original:", original)

changed:  [{'id': 1, 'age': 31}, {'id': 2, 'age': 26}]
original: [{'id': 1, 'age': 30}, {'id': 2, 'age': 25}]


In [197]:
# Totals and counts
total_age = 0
count = 0
for row in rows:
    total_age = total_age + row["age"]
    count = count + 1

print(f"total {total_age}, average {total_age / count}")

total 90, average 30.0


In [198]:
# Group and count, the pattern you will use constantly
by_city = {}
for row in rows:
    city = row["city"]
    if city in by_city:
        by_city[city] = by_city[city] + 1
    else:
        by_city[city] = 1

print(by_city)

{'Lagos': 2, 'Abuja': 1}


In [199]:

# SELECT currency, SUM(amount) FROM transactions GROUP BY currency

transactions = [
    {"id": 1, "user": "ahmed",  "amount": 5000,  "currency": "NGN", "status": "success"},
    {"id": 2, "user": "lahla",  "amount": 0,     "currency": "NGN", "status": "failed"},
    {"id": 3, "user": "judith", "amount": 120,   "currency": "USD", "status": "success"},
    {"id": 4, "user": "uche",   "amount": 300,   "currency": "NGN", "status": "pending"},
    {"id": 5, "user": "ahmed",  "amount": 80,    "currency": "USD", "status": "success"},
]


totals = {}

for tx in transactions:
    currency = tx["currency"]
    #print(currency)
    if currency in totals:
        totals[currency] = totals[currency] + tx["amount"]
           
    else:
        totals[currency] = tx["amount"]


#print(totals)
        #print(currency)
        #print(totals)

for currency, total in totals.items():
    print(f"{currency}: {total}")

NGN: 5300
USD: 200


In [200]:

# SELECT currency, SUM(amount) FROM transactions GROUP BY currency

transactions = [
    {"id": 1, "user": "ahmed",  "amount": 5000,  "currency": "NGN", "status": "success"},
    {"id": 2, "user": "lahla",  "amount": 0,     "currency": "NGN", "status": "failed"},
    {"id": 3, "user": "judith", "amount": 120,   "currency": "USD", "status": "success"},
    {"id": 4, "user": "uche",   "amount": 300,   "currency": "NGN", "status": "pending"},
    {"id": 5, "user": "ahmed",  "amount": 80,    "currency": "USD", "status": "success"},
    {"id": 6, "user": "judith", "amount": 120,   "currency": "EUR", "status": "success"},
    {"id": 7, "user": "uche",   "amount": 300,   "currency": "GBP", "status": "pending"},
    {"id": 8, "user": "ahmed",  "amount": 80,    "currency": "EUR", "status": "success"},
]


totals = set()
another_one = []

for tx in transactions:
    currency = tx["currency"]
    totals.add(currency)
    another_one.append(currency)

In [201]:
totals

{'EUR', 'GBP', 'NGN', 'USD'}

In [202]:
another_one

['NGN', 'NGN', 'USD', 'NGN', 'USD', 'EUR', 'GBP', 'EUR']

In [203]:
import pandas as pd

pd.DataFrame(rows)

,id,name,age,city,age_next_year
0,1,Ada,30,Lagos,31
1,2,Tunde,25,Abuja,26
2,3,Chi,35,Lagos,36


`pd.DataFrame(list_of_dicts)` just works. This is why a list of dictionaries is the
shape you aim for: it is one step from a DataFrame, and one step from a CSV, and one
step from a database insert.

---

## 3. Dict of dict: lookup by ID

Same records, but the outer container is a dictionary keyed by ID instead of a list.

Use this when you need to **find one record fast**, by its id.

In [204]:
by_id = {
    1: {"name": "Ada",   "age": 30, "city": "Lagos"},
    2: {"name": "Tunde", "age": 25, "city": "Abuja"},
    3: {"name": "Chi",   "age": 35, "city": "Lagos"},
}

print(f"{len(by_id)} records")
print(f"ids: {list(by_id.keys())}")

3 records
ids: [1, 2, 3]


In [205]:
# Access: key into the outer dict, then key into the inner dict
print(by_id[2])
print(by_id[2]["name"])

# Safe version, when the id might not exist
print(by_id.get(99))
print(by_id.get(99, {}).get("name"))       # None, and no crash

{'name': 'Tunde', 'age': 25, 'city': 'Abuja'}
Tunde
None
None


`by_id.get(99, {})` returns an empty dictionary when the id is missing, and calling
`.get("name")` on an empty dictionary is perfectly safe. Chaining `.get()` like this
is how you parse data that might be incomplete without a pile of `if` statements.

In [206]:
# Loop: .items() gives you the key AND the record
for user_id, user in by_id.items():
    print(user_id, user["name"], user["city"])

1 Ada Lagos
2 Tunde Abuja
3 Chi Lagos


In [207]:
# Filter
older = {}
for user_id, user in by_id.items():
    if user["age"] > 26:
        older[user_id] = user

print(older)

{1: {'name': 'Ada', 'age': 30, 'city': 'Lagos'}, 3: {'name': 'Chi', 'age': 35, 'city': 'Lagos'}}


In [208]:
# Select columns, keeping the id structure
names_only = {}
for user_id, user in by_id.items():
    names_only[user_id] = {"name": user["name"]}

print(names_only)

{1: {'name': 'Ada'}, 2: {'name': 'Tunde'}, 3: {'name': 'Chi'}}


In [209]:
# Modify and add
by_id[1]["age"] = 31
by_id[4] = {"name": "Musa", "age": 22, "city": "Kano"}

for user_id, user in by_id.items():
    print(user_id, user)

1 {'name': 'Ada', 'age': 31, 'city': 'Lagos'}
2 {'name': 'Tunde', 'age': 25, 'city': 'Abuja'}
3 {'name': 'Chi', 'age': 35, 'city': 'Lagos'}
4 {'name': 'Musa', 'age': 22, 'city': 'Kano'}


### Why bother, when a list of dicts holds the same data?

Because finding a record is instant. With a list you have to scan it.

In [210]:
import time

# Build the same data in both shapes
big_list = []
big_dict = {}
for i in range(200_000):
    record = {"id": i, "name": f"user_{i}"}
    big_list.append(record)
    big_dict[i] = record

target = 199_999

start = time.perf_counter()
found = None
for row in big_list:
    if row["id"] == target:
        found = row
        break
list_time = time.perf_counter() - start

start = time.perf_counter()
found = big_dict[target]
dict_time = time.perf_counter() - start

print(f"scanning the list: {list_time * 1000:8.3f} ms")
print(f"dict lookup:       {dict_time * 1000:8.3f} ms")
print()
print("The list has to check every record until it finds the match.")
print("The dict jumps straight to it, no matter how big the data gets.")

scanning the list:    6.673 ms
dict lookup:          0.015 ms

The list has to check every record until it finds the match.
The dict jumps straight to it, no matter how big the data gets.


In [211]:
import pandas as pd

by_id = {
    1: {"name": "Ada",   "age": 30},
    2: {"name": "Tunde", "age": 25},
}

print("orient='index' puts the keys down the side, as rows:")
print(pd.DataFrame.from_dict(by_id, orient="index"))

orient='index' puts the keys down the side, as rows:
    name  age
1    Ada   30
2  Tunde   25


In [212]:
print("plain pd.DataFrame() puts the keys ACROSS the top, as columns:")
print(pd.DataFrame(by_id))
print()
print("Almost never what you want with a dict of dict. Use from_dict(orient='index').")

plain pd.DataFrame() puts the keys ACROSS the top, as columns:
        1      2
name  Ada  Tunde
age    30     25

Almost never what you want with a dict of dict. Use from_dict(orient='index').


---

## 4. Dict of list: columnar

The container is a dictionary, and each value is a whole **column**.

This is how a spreadsheet thinks, and how pandas actually stores data underneath.

In [213]:
cols = {
    "id":   [1, 2, 3],
    "name": ["Ada", "Tunde", "Chi"],
    "age":  [30, 25, 35],
}

print(f"{len(cols)} columns")
print(f"{len(cols['id'])} rows")

3 columns
3 rows


In [214]:
# A whole column
print(cols["name"])

# One value: key for the column, then position for the row
print(cols["name"][1])         # Tunde
print(cols["age"][0])          # 30

['Ada', 'Tunde', 'Chi']
Tunde
30


**Careful with the order.** In a list of dicts you go `rows[1]["name"]`:
row first, then column. Here it is `cols["name"][1]`: column first, then row.
The same value, addressed the other way round.

In [215]:
# Loop one column
for name in cols["name"]:
    print(name)

Ada
Tunde
Chi


In [216]:
# Loop row by row: use the position to line the columns up
row_count = len(cols["id"])

for i in range(row_count):
    print(cols["id"][i], cols["name"][i], cols["age"][i])

1 Ada 30
2 Tunde 25
3 Chi 35


In [217]:
# Column totals are easy in this shape
total = 0
for age in cols["age"]:
    total = total + age

print(f"total age {total}, average {total / len(cols['age'])}")

total age 90, average 30.0


Column operations are easy here, row operations are awkward. That is exactly the
trade-off, and it is why pandas is fast at `df["age"].sum()` and slow at looping rows.

In [218]:
import pandas as pd

pd.DataFrame(cols)

,id,name,age
0,1,Ada,30
1,2,Tunde,25
2,3,Chi,35


`pd.DataFrame(dict_of_lists)` is the shape pandas likes best. Every column arrives
complete, so it can store each one as a single block of memory.

**One trap:** every list must be the same length, or pandas refuses.

In [219]:
bad = {"id": [1, 2, 3], "name": ["Ada", "Tunde"]}      # one short

try:
    pd.DataFrame(bad)
except ValueError as e:
    print(f"ValueError: {e}")

ValueError: All arrays must be of the same length


---

## 5. List of list: rows without names

A table where nobody told you the column names. This is what `csv.reader` gives you,
and what a database cursor returns.

In [220]:
data = [
    [1, "Ada",   30],
    [2, "Tunde", 25],
    [3, "Chi",   35],[1]
]

print(f"{len(data)} rows, {len(data[0])} columns")

4 rows, 3 columns


In [221]:
data

[[1, 'Ada', 30], [2, 'Tunde', 25], [3, 'Chi', 35], [1]]

In [222]:
# Access by position, twice: row, then column
print(data[1])           # the whole second row
print(data[1][1])        # Tunde
print(data[0][2])        # 30

[2, 'Tunde', 25]
Tunde
30


In [223]:
oge = []
for row in data:
    oge.extend(row)

In [224]:
oge

[1, 'Ada', 30, 2, 'Tunde', 25, 3, 'Chi', 35, 1]

In [225]:
 w = [1,2,3,[4,5,6],7]

In [226]:
w = [1,2,3,[4,5,6],7]
p = []
for i in w:
    if type(i).__name__ == "list":
        p.extend(i)
    else:
        p.append(i)

In [227]:
p 

[1, 2, 3, 4, 5, 6, 7]

In [228]:
# One column, by its position
names = []
for row in data:
    names.append(row[1])

print(names)

IndexError: list index out of range

In [ ]:
# Loop with unpacking, which is much more readable
for user_id, name, age in data:
    print(user_id, name, age)

`for user_id, name, age in data:` unpacks each inner list into three named
variables. Far better than `row[0]`, `row[1]`, `row[2]` scattered through your code.

It only works when every row has exactly that many items. One short row and you get
`ValueError: not enough values to unpack`.

In [ ]:
ragged = [[1, "Ada", 30], [2, "Tunde"]]

try:
    for user_id, name, age in ragged:
        print(user_id, name, age)
except ValueError as e:
    print(f"ValueError: {e}")

In [ ]:
# Give it column names by converting to a list of dicts
columns = ["id", "name", "age"]

rows = []
for row in data:
    record = {}
    position = 0
    for column in columns:
        record[column] = row[position]
        position = position + 1
    rows.append(record)

for record in rows:
    print(record)

That inner loop is doing something so common that Python has a built-in for it,
called `zip`. You will meet it in the shortcuts section below.

In [ ]:
import pandas as pd

print("without column names, you get 0, 1, 2:")
print(pd.DataFrame(data))
print()
print("with them:")
print(pd.DataFrame(data, columns=["id", "name", "age"]))

---

## 6. Nested dict inside a record

A field whose value is itself a dictionary. Very common in JSON.

In [ ]:
rows = [
    {"id": 1, "name": "Ada",   "address": {"city": "Lagos", "country": "NG"}},
    {"id": 2, "name": "Tunde", "address": {"city": "Abuja", "country": "NG"}},
    {"id": 3, "name": "Chi",   "address": {"city": "Accra", "country": "GH"}},
]

print(rows[0])

In [ ]:
# Access: keep going in, one key at a time
print(rows[0]["address"])                 # the inner dictionary
print(rows[0]["address"]["city"])         # a value inside it

In [ ]:
# Get the nested field for every row
cities = []
for row in rows:
    cities.append(row["address"]["city"])

print(cities)

In [ ]:
# When the nested part might be missing
messy = [
    {"id": 1, "address": {"city": "Lagos"}},
    {"id": 2},                                  # no address at all
    {"id": 3, "address": {}},                   # address, but empty
]

print("the unsafe way:")
for row in messy:
    try:
        print("  ", row["address"]["city"])
    except KeyError as e:
        print(f"   KeyError: {e}")

print()
print("the safe way:")
for row in messy:
    city = row.get("address", {}).get("city", "unknown")
    print("  ", row["id"], city)

In [ ]:
# Flatten it yourself, so every field is top level
flat = []
for row in rows:
    record = {
        "id": row["id"],
        "name": row["name"],
        "city": row["address"]["city"],
        "country": row["address"]["country"],
    }
    flat.append(record)

for record in flat:
    print(record)

In [ ]:
import pandas as pd

print("plain DataFrame leaves the dictionary sitting in a cell:")
print(pd.DataFrame(rows))
print()
print("json_normalize flattens it into address.city and address.country:")
print(pd.json_normalize(rows))

A dictionary sitting inside a DataFrame cell is useless: you cannot filter on it,
group by it, or write it to a warehouse column. Flatten nested data before it becomes
a DataFrame, either with your own loop or with `pd.json_normalize`.

---

## 7. Dict of list of dict: the API response

The shape almost every API returns. An outer dictionary of metadata, with the actual
rows tucked inside one key.

In [ ]:
response = {
    "status": "ok",
    "page": 1,
    "total": 3,
    "data": [
        {"id": 1, "name": "Ada",   "age": 30},
        {"id": 2, "name": "Tunde", "age": 25},
        {"id": 3, "name": "Chi",   "age": 35},
    ],
}

print(f"status: {response['status']}")
print(f"page:   {response['page']}")
print(f"rows:   {len(response['data'])}")

In [ ]:
# Read it from the outside in:
# response is {}        -> use a key
# response["data"] is []-> loop it
# each item is {}       -> use a key

print(response["data"])              # the list of rows
print(response["data"][0])           # the first row
print(response["data"][0]["name"])   # one field of it

In [ ]:
# Get a column out of it
names = []
for record in response["data"]:
    names.append(record["name"])

print(names)

In [ ]:
# The metadata is the part beginners forget to check
if response["status"] != "ok":
    print("the API is telling you something went wrong. stop here.")
else:
    print(f"processing {len(response['data'])} of {response['total']} record(s)")
    if len(response["data"]) < response["total"]:
        print("there are more pages to fetch")

**Always look at the metadata.** The `status`, `total`, `next` and `page` fields are
how the API tells you it failed, or that you only have a third of the data. Jumping
straight to `response["data"]` is how people end up loading 6 records out of 826 and
not noticing.

In [ ]:
import pandas as pd

pd.DataFrame(response["data"])

Note `pd.DataFrame(response["data"])`, not `pd.DataFrame(response)`. You hand pandas
the list of rows, not the envelope it arrived in.

---

## 8. A list inside each record: the deep mix

One record that owns several child records. An order with its line items, a customer
with their orders, a character with their episodes.

This is the shape people get stuck on, and the answer is always the same: **two
loops, outer then inner.**

In [ ]:
customers = [
    {"id": 1, "name": "Ada", "orders": [{"amt": 50}, {"amt": 20}, {"amt": 30}]},
    {"id": 2, "name": "Tunde", "orders": [{"amt": 10}]},
    {"id": 3, "name": "Chi", "orders": []},
]

print(customers[0])
print()
print(customers[0]["orders"])            # the inner list
print(customers[0]["orders"][1])         # the second order
print(customers[0]["orders"][1]["amt"])  # its amount

In [ ]:
# The two-loop pattern. Outer loop over customers, inner loop over their orders.
for customer in customers:
    print(f"{customer['name']}:")
    for order in customer["orders"]:
        print(f"   {order['amt']}")

In [ ]:
# Totals per parent
for customer in customers:
    total = 0
    for order in customer["orders"]:
        total = total + order["amt"]
    print(f"{customer['name']}: {len(customer['orders'])} order(s), total {total}")

Notice `total = 0` sits **inside** the outer loop. It has to reset for each
customer. Put it above the outer loop and you get a running total across everybody,
which is a different (and usually wrong) answer.

In [ ]:
# The wrong version, so you recognise it
total = 0                                 # outside. this is the bug
for customer in customers:
    for order in customer["orders"]:
        total = total + order["amt"]
    print(f"{customer['name']}: {total}")

print()
print("Ada's total is right, everyone after her is inflated by the people before.")

In [ ]:
# Flattening: one row per child, carrying the parent's details down
flat = []
for customer in customers:
    for order in customer["orders"]:
        flat.append({
            "customer_id": customer["id"],
            "customer_name": customer["name"],
            "amt": order["amt"],
        })

for row in flat:
    print(row)

print()
print(f"{len(customers)} customer(s) became {len(flat)} order row(s)")

**Chi disappeared.** She has no orders, so the inner loop never ran, so nothing was
appended for her. That is correct for an orders table, and it is also exactly how a
customer silently vanishes from a report.

If you need every customer represented, add the empty case explicitly.

In [ ]:
flat = []
for customer in customers:
    if len(customer["orders"]) == 0:
        flat.append({
            "customer_id": customer["id"],
            "customer_name": customer["name"],
            "amt": None,
        })
    else:
        for order in customer["orders"]:
            flat.append({
                "customer_id": customer["id"],
                "customer_name": customer["name"],
                "amt": order["amt"],
            })

for row in flat:
    print(row)

In [ ]:
import pandas as pd

print("plain DataFrame: the orders list sits in a cell, unusable")
print(pd.DataFrame(customers))
print()
print("json_normalize with record_path and meta: one row per order")
print(pd.json_normalize(customers, record_path="orders", meta=["id", "name"]))

`record_path="orders"` says "the rows I want are inside the orders list".
`meta=["id", "name"]` says "carry these parent fields down onto every child row".

That one call does exactly what your two loops did. Write the loops first until the
shape is obvious to you, then use `json_normalize` because it is shorter and handles
the awkward cases.

One difference to know: `json_normalize` drops parents with an empty list, just like
the first version of the loop did.

---

## Your turn: reading the shapes

Use plain loops. No comprehensions.

**Task 1.** Given `student = {"name": "Ada", "score": 78, "track": "data"}`, print the
name, print the score safely with `.get()`, add a `"level": 300` field, and print the
whole record one field per line.

**Task 2.** Given the table below, print every student's name, then print only the
students who scored above 70.

```python
students = [
    {"id": 1, "name": "Ada",   "score": 78, "track": "data"},
    {"id": 2, "name": "Tunde", "score": 64, "track": "web"},
    {"id": 3, "name": "Chi",   "score": 91, "track": "data"},
]
```

**Task 3.** Using the same table, count how many students are on each track and print
the result as a dictionary.

**Task 4.** Turn `students` into a dict of dict keyed by `id`, then print the name of
student 2 using the lookup.

**Task 5.** Given `cols = {"name": ["Ada", "Tunde"], "score": [78, 64]}`, print each
student on its own line as `"Ada scored 78"`.

**Task 6.** Given `data = [[1, "Ada", 78], [2, "Tunde", 64]]`, print the names, using
unpacking in the `for` line rather than `row[1]`.

**Task 7.** Given the nested records below, print each name with its city. Two records
are incomplete, so nothing may crash.

```python
people = [
    {"name": "Ada",   "address": {"city": "Lagos"}},
    {"name": "Tunde"},
    {"name": "Chi",   "address": {}},
]
```

**Task 8.** Given the API response below, check the status, then print each record's
name and how many orders it has.

```python
response = {
    "status": "ok",
    "data": [
        {"name": "Ada",   "orders": [{"amt": 50}, {"amt": 20}]},
        {"name": "Tunde", "orders": [{"amt": 10}]},
        {"name": "Chi",   "orders": []},
    ],
}
```

**Task 9.** Using that response, flatten it to one row per order, carrying the name
down. Print how many rows you ended up with, and say what happened to Chi.

**Task 10.** Still using that response, print each person's total order amount.
Remember where the running total has to be reset.

In [ ]:
# Task 1
student = {"name": "Ada", "score": 78, "track": "data"}


In [ ]:
# Tasks 2 and 3
students = [
    {"id": 1, "name": "Ada",   "score": 78, "track": "data"},
    {"id": 2, "name": "Tunde", "score": 64, "track": "web"},
    {"id": 3, "name": "Chi",   "score": 91, "track": "data"},
]


In [ ]:
# Task 4


In [ ]:
# Task 5
cols = {"name": ["Ada", "Tunde"], "score": [78, 64]}


In [ ]:
# Task 6
data = [[1, "Ada", 78], [2, "Tunde", 64]]


In [ ]:
# Task 7
people = [
    {"name": "Ada",   "address": {"city": "Lagos"}},
    {"name": "Tunde"},
    {"name": "Chi",   "address": {}},
]


In [ ]:
# Tasks 8, 9 and 10
response = {
    "status": "ok",
    "data": [
        {"name": "Ada",   "orders": [{"amt": 50}, {"amt": 20}]},
        {"name": "Tunde", "orders": [{"amt": 10}]},
        {"name": "Chi",   "orders": []},
    ],
}


---

## Drill: name that shape

Before writing any code against a new structure, say out loud what it is and how you
would reach one value inside it. Do these as a class, quickly.

For each one below, answer three questions:

1. What shape is it?
2. How do I get at the value shown?
3. Would `pd.DataFrame(thing)` work directly?

In [ ]:
mystery_1 = {"city": "Lagos", "population": 15000000}

mystery_2 = [{"city": "Lagos"}, {"city": "Abuja"}]

mystery_3 = {"lagos": {"population": 15000000}, "abuja": {"population": 3500000}}

mystery_4 = {"city": ["Lagos", "Abuja"], "population": [15000000, 3500000]}

mystery_5 = [["Lagos", 15000000], ["Abuja", 3500000]]

mystery_6 = {"results": [{"city": "Lagos", "districts": [{"name": "Ikeja"}]}]}

mystery_7 = [{"city": "Lagos", "coords": {"lat": 6.5, "lon": 3.3}}]

mystery_8 = {"ok": True, "data": {"cities": [{"name": "Lagos"}]}}

for i, thing in enumerate([mystery_1, mystery_2, mystery_3, mystery_4,
                           mystery_5, mystery_6, mystery_7, mystery_8], start=1):
    print(f"--- mystery_{i} ---")
    shape_of(thing)
    print()

**Now get the value.** Write the expression that returns each of these, without
running `shape_of` first. Check yourself afterwards.

1. The population from `mystery_1`
2. `"Abuja"` from `mystery_2`
3. Lagos's population from `mystery_3`
4. `"Abuja"` from `mystery_4`
5. `15000000` from `mystery_5`
6. `"Ikeja"` from `mystery_6`
7. The latitude from `mystery_7`
8. `"Lagos"` from `mystery_8`

In [ ]:
# Your answers here


---

## Converting between the shapes

This is the real skill. Data arrives in one shape and you need it in another.
Learn these six conversions and you can move between any of them.

In [ ]:
# The data, in list of dict form, as our starting point
rows = [
    {"id": 1, "name": "Ada",   "age": 30},
    {"id": 2, "name": "Tunde", "age": 25},
    {"id": 3, "name": "Chi",   "age": 35},
]

In [ ]:
# 1. List of dict  ->  Dict of dict  (build a lookup by id)
by_id = {}
for row in rows:
    by_id[row["id"]] = row

print(by_id)
print(by_id[2]["name"])

In [ ]:
# 2. Dict of dict  ->  List of dict  (put the key back in as a field)
back_to_rows = []
for user_id, user in by_id.items():
    record = dict(user)              # copy so we do not touch the original
    record["id"] = user_id
    back_to_rows.append(record)

for record in back_to_rows:
    print(record)

In [ ]:
# 3. List of dict  ->  Dict of list  (rows to columns)
cols = {}
for row in rows:
    for key, value in row.items():
        if key not in cols:
            cols[key] = []
        cols[key].append(value)

print(cols)

In [ ]:
# 4. Dict of list  ->  List of dict  (columns back to rows)
back = []
row_count = len(cols["id"])

for i in range(row_count):
    record = {}
    for key in cols:
        record[key] = cols[key][i]
    back.append(record)

for record in back:
    print(record)

In [ ]:
# 5. List of list  ->  List of dict  (give the columns names)
data = [[1, "Ada", 30], [2, "Tunde", 25]]
column_names = ["id", "name", "age"]

named = []
for row in data:
    record = {}
    position = 0
    for column in column_names:
        record[column] = row[position]
        position = position + 1
    named.append(record)

for record in named:
    print(record)

In [ ]:
# 6. List of dict  ->  List of list  (drop the names, keep the order)
plain = []
for row in rows:
    values = []
    for column in ["id", "name", "age"]:
        values.append(row[column])
    plain.append(values)

print(plain)

Notice conversion 6 loops over an explicit list of column names rather than over
`row.keys()`. That is deliberate: it guarantees every output row has its values in
the same order, even if one dictionary happens to list its keys differently.

When you are flattening to positional data, always name the columns yourself.

### The conversion map

| From | To | How |
|---|---|---|
| list of dict | dict of dict | loop, `out[row["id"]] = row` |
| dict of dict | list of dict | loop `.items()`, put the key back as a field |
| list of dict | dict of list | loop rows, then loop fields, append into per-key lists |
| dict of list | list of dict | loop by position, pull the same index from every column |
| list of list | list of dict | loop rows, pair values with your column names |
| list of dict | list of list | loop rows, pull the columns in a fixed order |
| anything nested | flat rows | two loops, outer then inner, carrying parent fields down |
| anything | DataFrame | get it to a list of dict first, then `pd.DataFrame(rows)` |

**The one to remember:** when in doubt, convert to a **list of dictionaries**. It is
the shape pandas wants, the shape a CSV writer wants, the shape a database insert
wants, and the shape a human can read.

---

# The shortcuts: `enumerate`, `zip`, `sorted`, `map`, `filter`

Everything up to here used plain loops, and plain loops will always work. This section
is about the built-in functions that replace the most common of those loops.

**Learn them in this order.** Each one is easy once the loop it replaces is familiar:

1. `enumerate` and `zip` (easy wins, no lambda needed)
2. `sorted`, `max`, `min`, `sum` with `key=`
3. `any` and `all`
4. `map` and `filter` (last, because they need `lambda`)

**One thing to know before you start.** `zip`, `map` and `filter` return **lazy
objects**, not lists. They do not compute anything until you ask. Wrap them in
`list()` or `dict()` to see the result.

In [ ]:
# Proof: they do not look like lists
keys = ["id", "name"]
values = [1, "Ada"]

print(zip(keys, values))                    # a zip object, not the answer
print(list(zip(keys, values)))              # now you can see it
print(dict(zip(keys, values)))              # or make it a dict directly

In [ ]:
# And they are used up after one pass. This catches everyone once.
paired = zip(keys, values)

print("first time: ", list(paired))
print("second time:", list(paired))         # empty. it is exhausted
print()
print("If you need it twice, save it: paired = list(zip(keys, values))")

---

## 1. `enumerate`: loop with a counter

The loop it replaces:

```python
i = 0
for row in rows:
    print(i, row["name"])
    i = i + 1
```

In [ ]:
rows = [
    {"id": 1, "name": "Ada",   "age": 30},
    {"id": 2, "name": "Tunde", "age": 25},
    {"id": 3, "name": "Chi",   "age": 35},
]

for i, row in enumerate(rows):
    print(i, row["name"])

In [ ]:
# start=1 when you are numbering for a human
for position, row in enumerate(rows, start=1):
    print(f"{position}. {row['name']}")

`enumerate` gives you two things each time round: the counter and the item.
Use it whenever you catch yourself writing `i = i + 1` at the bottom of a loop.

It is also how you report a line number when a row fails validation, which is the
difference between "a row was bad" and "row 4,812 was bad".

In [ ]:
messy = [
    {"id": 1, "amount": "100"},
    {"id": 2, "amount": "abc"},
    {"id": 3, "amount": ""},
]

for line_number, row in enumerate(messy, start=2):      # start=2, line 1 is the header
    try:
        amount = float(row["amount"])
        print(f"line {line_number}: ok, {amount}")
    except ValueError:
        print(f"line {line_number}: bad amount {row['amount']!r}")

---

## 2. `zip`: pair things up, side by side

`zip` walks several lists together and hands you one item from each, per round.

The loop it replaces:

```python
i = 0
for name in names:
    print(name, ages[i])
    i = i + 1
```

In [ ]:
names = ["Ada", "Tunde", "Chi"]
ages = [30, 25, 35]

for name, age in zip(names, ages):
    print(name, age)

In [ ]:
# Three at once works exactly the same way
ids = [1, 2, 3]

for user_id, name, age in zip(ids, names, ages):
    print(user_id, name, age)

### `zip` + `dict` is the single most useful trick here

`dict(zip(keys, values))` builds a record from a list of column names and a list of
values. That is the "list of list to list of dict" conversion, in one line.

In [ ]:
columns = ["id", "name", "age"]
values = [1, "Ada", 30]

print(dict(zip(columns, values)))

In [ ]:
# The whole conversion, replacing the inner loop from earlier
data = [[1, "Ada", 30], [2, "Tunde", 25], [3, "Chi", 35]]
columns = ["id", "name", "age"]

rows = []
for row in data:
    rows.append(dict(zip(columns, row)))

for record in rows:
    print(record)

Compare that with the version further up, which needed a `position` counter and an
inner loop. Same result, three lines shorter, and no index to get wrong.

This is exactly what `csv.DictReader` does internally: it zips the header row against
every data row.

In [ ]:
# Dict of list -> list of dict, using zip on the values
cols = {"id": [1, 2, 3], "name": ["Ada", "Tunde", "Chi"]}

rows = []
for user_id, name in zip(cols["id"], cols["name"]):
    rows.append({"id": user_id, "name": name})

for record in rows:
    print(record)

### `zip` stops at the shortest

This is a feature and a trap. No error, it just quietly stops.

In [ ]:
names = ["Ada", "Tunde", "Chi"]
ages = [30, 25]                              # one short

for name, age in zip(names, ages):
    print(name, age)

print()
print("Chi is missing, and nothing warned you.")
print()
print("Guard it yourself when the lengths should match:")
if len(names) != len(ages):
    print(f"MISMATCH: {len(names)} names but {len(ages)} ages")

In [ ]:
# Python 3.10+ can complain for you
try:
    for name, age in zip(names, ages, strict=True):
        print(name, age)
except ValueError as e:
    print(f"ValueError: {e}")

### `zip(*rows)`: flipping rows into columns

The `*` unpacks a list of rows into separate arguments, so `zip` pairs them
column-wise instead. This transposes your data.

In [ ]:
data = [[1, "Ada", 30], [2, "Tunde", 25], [3, "Chi", 35]]

print("rows:")
for row in data:
    print("  ", row)

print()
print("transposed into columns:")
for column in zip(*data):
    print("  ", column)

In [ ]:
# Which gives you the list of list -> dict of list conversion
columns = ["id", "name", "age"]
cols = dict(zip(columns, zip(*data)))

print(cols)
print()
print("the values are tuples. make them lists if you need to append later:")
cols = {}
for name, values in zip(columns, zip(*data)):
    cols[name] = list(values)

print(cols)

---

## 3. `sorted`, `max`, `min`, `sum` with `key=`

Sorting a list of dictionaries needs one extra piece of information: **which field**
to sort by. That is what `key=` is for.

In [ ]:
rows = [
    {"name": "Ada",   "age": 30, "city": "Lagos"},
    {"name": "Tunde", "age": 25, "city": "Abuja"},
    {"name": "Chi",   "age": 35, "city": "Lagos"},
]

# Without a key, Python does not know what to compare
try:
    sorted(rows)
except TypeError as e:
    print(f"TypeError: {e}")

In [ ]:
# key= says "when comparing two rows, compare their ages"
youngest_first = sorted(rows, key=lambda row: row["age"])

for row in youngest_first:
    print(row["name"], row["age"])

`lambda row: row["age"]` is a tiny unnamed function. Read it as:
*given a row, give me its age*. Python calls it once per item and sorts by whatever
comes back.

If `lambda` feels like too much for your students on day one, the same thing works
with a normal function.

In [ ]:
def get_age(row):
    """Given a row, return its age. Used as a sort key."""
    return row["age"]

for row in sorted(rows, key=get_age):
    print(row["name"], row["age"])

In [ ]:
# Reverse, and sorting by two fields at once
print("oldest first:")
for row in sorted(rows, key=lambda r: r["age"], reverse=True):
    print("  ", row["name"], row["age"])

print()
print("by city, then by age within each city:")
for row in sorted(rows, key=lambda r: (r["city"], r["age"])):
    print("  ", row["city"], row["name"], row["age"])

Returning a tuple from the key function sorts by the first item, then breaks ties
with the second. `(r["city"], r["age"])` is `ORDER BY city, age`.

In [ ]:
# max and min take the same key, and give you the whole record back
oldest = max(rows, key=lambda r: r["age"])
youngest = min(rows, key=lambda r: r["age"])

print("oldest: ", oldest)
print("youngest:", youngest)
print()
print("Note you get the RECORD, not just the age. That is usually what you wanted.")

In [ ]:
# sum has no key, so pull the column out first
ages = []
for row in rows:
    ages.append(row["age"])

print(f"total {sum(ages)}, average {sum(ages) / len(ages)}")
print(f"range: {min(ages)} to {max(ages)}")

`max(rows, key=...)` gives you the record with the biggest age.
`max(ages)` gives you the biggest age itself. Two different questions, and mixing
them up is a common bug.

Also worth knowing: `max` of an empty list raises `ValueError`. Pass a default when
the list might be empty.

In [ ]:
empty = []

try:
    max(empty)
except ValueError as e:
    print(f"ValueError: {e}")

print(max(empty, default=0))
print(sum(empty))                  # sum is fine with empty, it gives 0

---

## 4. `any` and `all`: quick checks over a whole list

In [ ]:
rows = [
    {"name": "Ada",   "age": 30, "email": "ada@example.com"},
    {"name": "Tunde", "age": 25, "email": ""},
    {"name": "Chi",   "age": 35, "email": "chi@example.com"},
]

# Build a list of True/False, then ask a question about it
has_email = []
for row in rows:
    has_email.append(row["email"] != "")

print(has_email)
print(f"does everyone have an email? {all(has_email)}")
print(f"does anyone have an email?  {any(has_email)}")
print(f"how many?                   {sum(has_email)}")

`sum()` on a list of `True`/`False` counts the `True`s, because `True` is 1 and
`False` is 0. That is the same trick as `df.isna().sum()` in pandas.

---

## 5. `map` and `filter`: last, because they need `lambda`

These two replace the two most common loops of all. Teach them last, and be honest
with students that a plain loop is often more readable.

### `map`: do something to every item

The loop it replaces:

```python
names = []
for row in rows:
    names.append(row["name"])
```

In [ ]:
rows = [
    {"id": 1, "name": "Ada",   "age": 30},
    {"id": 2, "name": "Tunde", "age": 25},
    {"id": 3, "name": "Chi",   "age": 35},
]

names = list(map(lambda row: row["name"], rows))
print(names)

In [ ]:
# map with an existing function needs no lambda at all
shouted = list(map(str.upper, names))
print(shouted)

lengths = list(map(len, names))
print(lengths)

numbers_as_text = ["10", "20", "30"]
print(list(map(int, numbers_as_text)))

`list(map(int, values))` is the tidiest way to convert a whole list of strings to
numbers. No lambda, no loop, and it reads as "apply int to each one".

In [ ]:
# map over two lists at once, pairing them like zip
ids = [1, 2, 3]
names = ["Ada", "Tunde", "Chi"]

labels = list(map(lambda i, n: f"{i}: {n}", ids, names))
for label in labels:
    print(label)

### `filter`: keep the items that pass a test

The loop it replaces:

```python
older = []
for row in rows:
    if row["age"] > 26:
        older.append(row)
```

In [ ]:
older = list(filter(lambda row: row["age"] > 26, rows))

for row in older:
    print(row["name"], row["age"])

In [ ]:
# filter on a dict of dict: .items() gives (key, value) pairs,
# so item[0] is the id and item[1] is the record
by_id = {
    1: {"name": "Ada",   "age": 30},
    2: {"name": "Tunde", "age": 25},
    3: {"name": "Chi",   "age": 35},
}

older = dict(filter(lambda item: item[1]["age"] > 26, by_id.items()))
print(older)

That `item[1]["age"]` is exactly why plain loops win for beginners. Compare:

```python
older = {}
for user_id, user in by_id.items():
    if user["age"] > 26:
        older[user_id] = user
```

Four lines, and every one of them says what it means. Use `filter` when the condition
is short and obvious, and a loop when anyone would have to stop and decode it.

In [ ]:
# filter with no lambda drops the falsy items: 0, "", None, empty list, empty dict
mixed = [1, 0, "Ada", "", None, [], "Tunde", 5]

print(list(filter(None, mixed)))

In [ ]:
# Chaining them: filter first, then map, reads right to left and hurts
names_of_older = list(map(lambda r: r["name"],
                          filter(lambda r: r["age"] > 26, rows)))
print(names_of_older)

print()
print("the loop version, which you can read top to bottom:")
names_of_older = []
for row in rows:
    if row["age"] > 26:
        names_of_older.append(row["name"])
print(names_of_older)

### The honest summary

| Function | Replaces | Teach it |
|---|---|---|
| `enumerate` | a manual counter | early, it is a pure win |
| `zip` | walking two lists by index | early, especially `dict(zip(...))` |
| `sorted(key=)` | there is no simple loop version | when they need sorting |
| `max`/`min(key=)` | a loop tracking the best so far | alongside sorted |
| `any`/`all`/`sum` | a loop with a flag or counter | alongside those |
| `map` | a loop that appends a transformed value | late |
| `filter` | a loop with an `if` that appends | late |

`enumerate` and `zip` make code clearer. `map` and `filter` make it shorter, which is
not always the same thing.

---

## Your turn: the shortcuts

**Task 1.** Given `names = ["Ada", "Tunde", "Chi"]`, print each name numbered from 1,
using `enumerate`.

**Task 2.** Given `columns = ["id", "name", "score"]` and `values = [1, "Ada", 78]`,
build a record dictionary in one line with `zip`.

**Task 3.** Given `data = [[1, "Ada", 78], [2, "Tunde", 64]]` and the same column
names, convert the whole thing to a list of dicts using `zip` inside a loop.

**Task 4.** Given `cols = {"id": [1, 2], "name": ["Ada", "Tunde"]}`, use `zip` to
print each row on its own line.

**Task 5.** Sort the students table by score, highest first, and print the top two.

**Task 6.** Print the record of the student with the highest score, using `max` with a
key. Then print just the highest score itself.

**Task 7.** Sort the students by track, then by score within each track.

**Task 8.** Using `any` and `all`, check whether every student scored above 50, and
whether any scored above 90.

**Task 9.** Use `map` to get a list of the names, and `filter` to get the students who
passed. Remember to wrap both in `list()`.

**Task 10.** Take `raw = ["10", "20", "30"]` and convert it to a list of integers with
`map`, with no lambda.

**Task 11 (thinking task).** Take the flatten from shape 8 and write it with plain
loops, then look at your answer and say which of the shortcuts, if any, would make it
clearer. There is a right answer and it might be "none of them".

In [ ]:
# Tasks 1 to 4
students = [
    {"id": 1, "name": "Ada",   "score": 78, "track": "data"},
    {"id": 2, "name": "Tunde", "score": 64, "track": "web"},
    {"id": 3, "name": "Chi",   "score": 91, "track": "data"},
]
names = ["Ada", "Tunde", "Chi"]
columns = ["id", "name", "score"]
values = [1, "Ada", 78]
data = [[1, "Ada", 78], [2, "Tunde", 64]]
cols = {"id": [1, 2], "name": ["Ada", "Tunde"]}


In [ ]:
# Tasks 5, 6 and 7


In [ ]:
# Tasks 8, 9 and 10
raw = ["10", "20", "30"]


In [ ]:
# Task 11


---

## Every shape, into pandas

One table to keep. The right call depends entirely on the shape you are holding.

In [ ]:
import pandas as pd

one_record  = {"id": 1, "name": "Ada"}
rows        = [{"id": 1, "name": "Ada"}, {"id": 2, "name": "Tunde"}]
by_id       = {1: {"name": "Ada"}, 2: {"name": "Tunde"}}
cols        = {"id": [1, 2], "name": ["Ada", "Tunde"]}
plain       = [[1, "Ada"], [2, "Tunde"]]
nested      = [{"id": 1, "address": {"city": "Lagos"}}]
api         = {"status": "ok", "data": [{"id": 1, "name": "Ada"}]}
deep        = [{"id": 1, "orders": [{"amt": 50}, {"amt": 20}]}]

print("1. one dict          ->", list(pd.DataFrame([one_record]).columns))
print("2. list of dict      ->", list(pd.DataFrame(rows).columns))
print("3. dict of dict      ->", list(pd.DataFrame.from_dict(by_id, orient="index").columns))
print("4. dict of list      ->", list(pd.DataFrame(cols).columns))
print("5. list of list      ->", list(pd.DataFrame(plain, columns=["id", "name"]).columns))
print("6. nested dict       ->", list(pd.json_normalize(nested).columns))
print("7. api response      ->", list(pd.DataFrame(api["data"]).columns))
print("8. list in records   ->", list(pd.json_normalize(deep, record_path="orders",
                                                       meta=["id"]).columns))

| Shape | The call |
|---|---|
| one dict | `pd.DataFrame([d])` (note the brackets) |
| list of dict | `pd.DataFrame(rows)` |
| dict of dict | `pd.DataFrame.from_dict(by_id, orient="index")` |
| dict of list | `pd.DataFrame(cols)` |
| list of list | `pd.DataFrame(data, columns=[...])` |
| nested dict | `pd.json_normalize(rows)` |
| API response | `pd.DataFrame(response["data"])` |
| list inside records | `pd.json_normalize(rows, record_path="orders", meta=["id"])` |

Two to watch:

- `pd.DataFrame(one_dict)` without brackets raises. Wrap it in a list.
- `pd.DataFrame(dict_of_dict)` runs, but puts your ids across the top as columns.
  Use `from_dict(..., orient="index")`.

In [ ]:
# json_normalize on an API response: point it at the list, not the envelope
api = {
    "status": "ok",
    "data": [
        {"id": 1, "name": "Ada", "address": {"city": "Lagos"}},
        {"id": 2, "name": "Tunde", "address": {"city": "Abuja"}},
    ],
}

print(pd.json_normalize(api["data"]))
print()
print("or let it walk in for you with record_path:")
print(pd.json_normalize(api, record_path="data"))

---

## Mini project: one dataset, five shapes

You are handed an API response. Get it into every shape, using plain loops, then
answer some questions about it.

In [ ]:
response = {
    "status": "ok",
    "page": 1,
    "total": 5,
    "data": [
        {"id": 101, "name": "Ada",   "city": "Lagos",
         "orders": [{"item": "laptop", "amt": 450000}, {"item": "mouse", "amt": 8000}]},
        {"id": 102, "name": "Tunde", "city": "Abuja",
         "orders": [{"item": "phone", "amt": 180000}]},
        {"id": 103, "name": "Chi",   "city": "Lagos",
         "orders": [{"item": "monitor", "amt": 95000}, {"item": "keyboard", "amt": 15000},
                    {"item": "headset", "amt": 22000}]},
        {"id": 104, "name": "Musa",  "city": "Kano",
         "orders": []},
        {"id": 105, "name": "Ngozi", "city": "Lagos",
         "orders": [{"item": "laptop", "amt": 520000}]},
    ],
}

shape_of(response)

### What to build

Plain loops for 1 to 8. The shortcuts are allowed from 9 onwards.

1. **Check the envelope.** Print the status, and warn if `len(data)` does not match
   `total`.

2. **A list of dict** of just the customers, with `id`, `name`, `city` and
   `order_count`. No orders list inside it.

3. **A dict of dict** keyed by `id`, so you can look up customer 103 instantly.
   Print Chi's city using the lookup.

4. **A dict of list** (columnar) with `id`, `name` and `city`. Print the `name`
   column, then print the second row by pulling index 1 from each column.

5. **A list of list**, values only, in the fixed order `id, name, city`. Print it.

6. **Flatten the orders**: one row per order, each carrying `customer_id`,
   `customer_name`, `city`, `item` and `amt`. Print the row count, and explain what
   happened to Musa.

7. **Flatten again**, this time keeping Musa with `item` and `amt` set to `None`.

8. **Totals.** Print each customer's order count and total spend. Then print the
   total revenue per city.

9. **Using the shortcuts:** sort the customers by total spend, highest first, and
   print the top three with `enumerate` numbering them from 1.

10. **Using the shortcuts:** find the single biggest order across everybody, with
    `max` and a key, and print the customer name, the item and the amount.

11. **pandas:** build a `customers` DataFrame and an `orders` DataFrame from your
    flattened rows, and confirm the total in the orders frame matches your step 8
    figure.

### Stretch

12. Write `to_rows(response)` that takes any response in this shape and returns the
    flattened order rows, so the whole thing is one reusable function.
13. Write `group_sum(rows, group_key, value_key)` that returns a dictionary of totals,
    and use it for both `city` and `item`.
14. Do step 6 with `pd.json_normalize` and confirm you get the same row count as your
    loop version.

In [ ]:
# 1 and 2



In [ ]:
# 3 and 4



In [ ]:
# 5



In [ ]:
# 6 and 7



In [ ]:
# 8



In [ ]:
# 9 and 10



In [ ]:
# 11



---

# Appendix: list comprehensions

**Skip this on day one.** It is here so students recognise the syntax when they meet
it in someone else's code, including the other notebooks in this course.

A comprehension is a loop that builds a list, written on one line:

```
[ what_to_keep   for item in collection   if condition ]
```

Read it in that order: the middle first (the loop), then the end (the filter), then
the start (what goes in the list).

In [ ]:
rows = [
    {"id": 1, "name": "Ada",   "age": 30},
    {"id": 2, "name": "Tunde", "age": 25},
    {"id": 3, "name": "Chi",   "age": 35},
]

# The loop you already know
names = []
for row in rows:
    names.append(row["name"])

# The same thing, one line
names_short = [row["name"] for row in rows]

print(names)
print(names_short)
print(names == names_short)

In [ ]:
# With a filter on the end
older = []
for row in rows:
    if row["age"] > 26:
        older.append(row["name"])

older_short = [row["name"] for row in rows if row["age"] > 26]

print(older, older_short)

In [ ]:
# Dict comprehension: same idea, building a dictionary
by_id = {}
for row in rows:
    by_id[row["id"]] = row

by_id_short = {row["id"]: row for row in rows}

print(list(by_id) == list(by_id_short))

In [ ]:
# Nested: the two-loop flatten, on one line
customers = [
    {"id": 1, "name": "Ada", "orders": [{"amt": 50}, {"amt": 20}]},
    {"id": 2, "name": "Tunde", "orders": [{"amt": 10}]},
]

flat = []
for customer in customers:
    for order in customer["orders"]:
        flat.append({"id": customer["id"], "amt": order["amt"]})

flat_short = [
    {"id": customer["id"], "amt": order["amt"]}
    for customer in customers
    for order in customer["orders"]
]

print(flat == flat_short)
print(flat_short)

The loops in a nested comprehension go in the **same order** as the nested loops
they replace: outer first, inner second. That trips people up because the body of the
loop appears first, before either of them.

**When not to use one.** If it no longer fits on one line, or it has two conditions
and a nested loop, write the ordinary loop. Clever is not the goal. Readable is.

---

## Common mistakes

| Mistake | What you see | The fix |
|---|---|---|
| `rows["name"]` on a list of dicts | `TypeError: list indices must be integers` | Loop it, or index first: `rows[0]["name"]` |
| `row[0]` on a dictionary | `KeyError: 0` | Dicts use keys, not positions |
| `d["email"]` on a missing key | `KeyError` | `.get("email")`, or check `in` first |
| `row["a"]["b"]` when `a` is missing | `KeyError` | `row.get("a", {}).get("b")` |
| `pd.DataFrame(one_dict)` | `ValueError` about scalars | Wrap it: `pd.DataFrame([d])` |
| `pd.DataFrame(dict_of_dict)` | Your ids become columns | `from_dict(..., orient="index")` |
| Nested dict left in a DataFrame | A dict sitting in a cell, unusable | `pd.json_normalize` |
| Running total declared outside the outer loop | Every total after the first is inflated | Reset it inside the outer loop |
| Flattening a parent with an empty list | That parent vanishes silently | Handle the empty case explicitly |
| Modifying `row` in a loop | The original list changes too | `dict(row)` to copy first |
| `zip` with lists of different lengths | Silently stops at the shortest | Check the lengths, or `strict=True` |
| Using a `zip`/`map`/`filter` twice | The second time is empty | Wrap in `list()` once and reuse |
| Printing `zip(...)` | `<zip object at 0x...>` | `list(zip(...))` or `dict(zip(...))` |
| `sorted(rows)` with no key | `TypeError: '<' not supported between dicts` | `sorted(rows, key=lambda r: r["age"])` |
| `max(ages)` when you wanted the record | You get a number, not a row | `max(rows, key=lambda r: r["age"])` |
| `max([])` | `ValueError` | `max(values, default=0)` |
| `cols["name"][1]` vs `rows[1]["name"]` | Right value, wrong shape assumed | Columnar is column-then-row, rows are row-then-column |

In [ ]:
# The two type errors, side by side, so students recognise them instantly
rows = [{"name": "Ada"}, {"name": "Tunde"}]
one = {"name": "Ada"}

try:
    rows["name"]
except TypeError as e:
    print(f"list of dicts, used like a dict -> TypeError: {e}")

try:
    one[0]
except KeyError as e:
    print(f"dict, used like a list          -> KeyError: {e}")

print()
print("The error tells you which shape you actually have.")
print("TypeError about indices means you are holding a LIST.")
print("KeyError means you are holding a DICT and that key is not in it.")

---

## The one-page summary

```
{}            one record          use a key            d["name"]
[{}, {}]      a table             loop it              for row in rows: row["name"]
{1: {}, 2:{}} lookup by id        key then key         by_id[2]["name"]
{"a": [], }   columns             key then position    cols["name"][1]
[[], []]      rows, no names      position twice       data[1][1]
[{"a": {}}]   nested              keep going in        rows[0]["a"]["b"]
{"data": []}  API response        key, then loop       response["data"][0]["name"]
[{"o": []}]   parent and children two loops            for r in rows: for o in r["o"]:
```

**Read from the outside in. `{` means use a key. `[` means use a position or loop.**

When in doubt, convert it to a list of dictionaries. Everything downstream wants that
shape.